# Phase 2: Patient ID Alignment
Find and align patient IDs across clinical, follow-up, and expression datasets.

In [1]:
import pandas as pd

# Load all three datasets
clinical = pd.read_csv("../data/clinical.tsv", sep="\t")
follow_up = pd.read_csv("../data/follow_up.tsv", sep="\t")
expression = pd.read_csv("../data/kirc_expression.tsv", sep="\t")

In [2]:
# 1. Find likely patient ID columns in clinical data
# Look for columns with "id", "submitter", "case", or "patient" in the name
clinical_id_cols = [c for c in clinical.columns if any(kw in c.lower() for kw in ["submitter_id", "case_id", "patient_id", "barcode"])]
print("Clinical ID columns:", clinical_id_cols)

Clinical ID columns: ['cases.case_id', 'cases.submitter_id', 'demographic.submitter_id', 'diagnoses.submitter_id', 'treatments.submitter_id']


In [3]:
# 2. Find likely patient ID columns in follow-up data
followup_id_cols = [c for c in follow_up.columns if any(kw in c.lower() for kw in ["submitter_id", "case_id", "patient_id", "barcode"])]
print("Follow-up ID columns:", followup_id_cols)

Follow-up ID columns: ['cases.case_id', 'cases.submitter_id', 'follow_ups.submitter_id', 'molecular_tests.submitter_id', 'other_clinical_attributes.submitter_id']


In [4]:
# 3. Print 5 example values from each ID column
print("=== Clinical ===")
for col in clinical_id_cols:
    print(f"  {col}: {clinical[col].head().tolist()}")

print("\n=== Follow-Up ===")
for col in followup_id_cols:
    print(f"  {col}: {follow_up[col].head().tolist()}")

=== Clinical ===
  cases.case_id: ['0022478c-4dfd-4cbe-a05e-fb20310844e3', '0022478c-4dfd-4cbe-a05e-fb20310844e3', '0022478c-4dfd-4cbe-a05e-fb20310844e3', '01277e9d-a35f-45d9-9e60-2e8cd79630a0', '01277e9d-a35f-45d9-9e60-2e8cd79630a0']
  cases.submitter_id: ['TCGA-B0-5088', 'TCGA-B0-5088', 'TCGA-B0-5088', 'TCGA-DV-A4W0', 'TCGA-DV-A4W0']
  demographic.submitter_id: ['TCGA-B0-5088_demographic', 'TCGA-B0-5088_demographic', 'TCGA-B0-5088_demographic', 'TCGA-DV-A4W0_demographic', 'TCGA-DV-A4W0_demographic']
  diagnoses.submitter_id: ['TCGA-B0-5088_diagnosis2', 'TCGA-B0-5088_diagnosis2', 'TCGA-B0-5088_diagnosis', 'TCGA-DV-A4W0_diagnosis2', 'TCGA-DV-A4W0_diagnosis2']
  treatments.submitter_id: ['TCGA-B0-5088_treatment2', 'TCGA-B0-5088_treatment', "'--", 'TCGA-DV-A4W0_treatment6', 'TCGA-DV-A4W0_treatment8']

=== Follow-Up ===
  cases.case_id: ['0022478c-4dfd-4cbe-a05e-fb20310844e3', '0022478c-4dfd-4cbe-a05e-fb20310844e3', '0022478c-4dfd-4cbe-a05e-fb20310844e3', '0022478c-4dfd-4cbe-a05e-fb203108

In [10]:
# 4. Inspect expression column names
# Columns are TCGA barcodes like TCGA-XX-XXXX-01 (samples)
# The first column(s) may be gene identifiers
non_tcga = [c for c in expression.columns if not c.startswith("TCGA")]
tcga_cols = [c for c in expression.columns if c.startswith("TCGA")]

print(f"Non-TCGA columns (gene IDs): {non_tcga}")
print(f"TCGA sample columns: {len(tcga_cols)}")
print(f"First 5 barcodes: {tcga_cols[:5]}")

Non-TCGA columns (gene IDs): ['sample']
TCGA sample columns: 606
First 5 barcodes: ['TCGA-BP-4162-01', 'TCGA-CJ-5677-11', 'TCGA-DV-5566-01', 'TCGA-BP-5191-01', 'TCGA-BP-5200-01']


In [11]:
# 5. Extract patient IDs from expression barcodes
# TCGA-XX-XXXX-01  →  keep first 3 parts  →  TCGA-XX-XXXX
# The last part (-01, -11) is sample type, not part of the patient ID
expr_patient_ids = list(set("-".join(c.split("-")[:3]) for c in tcga_cols))

print(f"Example patient IDs extracted: {sorted(expr_patient_ids)[:5]}")

Example patient IDs extracted: ['TCGA-3Z-A93Z', 'TCGA-6D-AA2E', 'TCGA-A3-3306', 'TCGA-A3-3307', 'TCGA-A3-3308']


In [13]:
# 6. Count unique patient IDs in each dataset
# Use cases.submitter_id — this is the TCGA barcode (e.g. TCGA-XX-XXXX)
# NOT cases.case_id, which is a UUID and won't match expression barcodes
clin_id_col = "cases.submitter_id"
fu_id_col   = "cases.submitter_id"

n_clin = clinical[clin_id_col].nunique()
n_fu   = follow_up[fu_id_col].nunique()
n_expr = len(expr_patient_ids)

print("Unique patients per dataset:")
print(f"  Clinical  ({clin_id_col}): {n_clin}")
print(f"  Follow-up ({fu_id_col}):   {n_fu}")
print(f"  Expression (from barcodes):  {n_expr}")

Unique patients per dataset:
  Clinical  (cases.submitter_id): 537
  Follow-up (cases.submitter_id):   537
  Expression (from barcodes):  533


In [14]:
# Overlap check using cases.submitter_id (TCGA barcodes)
clin_set = set(clinical[clin_id_col])
fu_set   = set(follow_up[fu_id_col])
expr_set = set(expr_patient_ids)

overlap_all = clin_set & fu_set & expr_set

print(f"Patients in clinical ∩ follow-up:   {len(clin_set & fu_set)}")
print(f"Patients in clinical ∩ expression:   {len(clin_set & expr_set)}")
print(f"Patients in follow-up ∩ expression:  {len(fu_set & expr_set)}")
print(f"Patients in ALL three datasets:      {len(overlap_all)}")

Patients in clinical ∩ follow-up:   537
Patients in clinical ∩ expression:   533
Patients in follow-up ∩ expression:  533
Patients in ALL three datasets:      533
